Environment Setup and Google Drive Mounting

In [ ]:
!pip install -q ultralytics

import os
import zipfile
import yaml
from google.colab import drive
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 7.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Google Drive Mounting

In [ ]:
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


Dataset Location and Extraction

In [ ]:
zip_options = [
    '/content/drive/MyDrive/Offline Arabic ADAS.zip',
    '/content/drive/MyDrive/Offline Arabic ADAS.yolov11.zip'
]

zip_path = None
for option in zip_options:
    if os.path.exists(option):
        zip_path = option
        break

if zip_path:
    print(f"Dataset found at: {zip_path}")
    print("Extracting dataset (8,842 images) to Colab local directory...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("Dataset successfully extracted to 'dataset' directory.")
else:
    raise FileNotFoundError("Compressed dataset not found in Drive. Ensure it is named 'Offline Arabic ADAS.zip'!")

Dataset found at: /content/drive/MyDrive/Offline Arabic ADAS.zip
Extracting dataset (8,842 images) to Colab local directory...
Dataset successfully extracted to 'dataset' directory.


Dataset Configuration and Path Initialization

In [ ]:
dataset_path = os.path.abspath('dataset')
yaml_path = None

# Recursively search for the dataset configuration file.
for root, dirs, files_list in os.walk(dataset_path):
    if 'data.yaml' in files_list:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path:
    print(f"Dataset configuration found: {yaml_path}")
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Update the dataset root path to match the extracted Colab directory.
    data_config['path'] = os.path.dirname(yaml_path)
    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f)
else:
    raise FileNotFoundError("Configuration file data.yaml not found inside extracted dataset!")

Dataset configuration found: /content/dataset/data.yaml


YOLO26 Nano Model Initialization

In [ ]:
print("Initializing YOLO26 Nano training...")

model = YOLO("yolo26n.pt")

Initializing YOLO26 Nano training...


YOLO26 Nano Training with Small-Target Detection Parameters

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    warmup_epochs=3,

    # Prevent early stopping before the scheduled training duration.
    patience=100,

    # Parameters used to improve detection of small and distant objects.
    multi_scale=True,
    scale=0.6,
    box=8.5,
    cls=1.5,

    # Fixed seed and deterministic execution for reproducible experiments.
    seed=42,
    deterministic=True,

    save_period=5,
    project="/content/drive/MyDrive/ADAS_YOLOv26_Model",
    name="vision_yolo26_scratch_100epochs",
    plots=True
)

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=vision_yolo26_scratch_100epochs, nbs=64, nms=No

Training Status

In [ ]:
print("\nTraining pipeline initiated successfully. YOLO26 Nano is now training on the dataset.")